## Time Series Modeling

### 1. Inspect dataframe

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Load data and set day as frequency
df = pd.read_parquet("cleaned_data.parquet")
df = df.asfreq("D")

In [ ]:
#Create new dataframe and sort
ts_df = df.copy()
ts_df.sort_index()

#Check the size and shape
print(ts_df.head(2))
print(ts_df.tail(2))

print("Start date:", ts_df.index.min())
print("End date:", ts_df.index.max())
print("Number of rows:", len(ts_df))

Our Time Series starts in 1990-01-01 and ends in 2025-12-31.

In [ ]:
#Check NA and NAN
print(ts_df.isna().sum())

We will remove sunshine_duration since it has missing values and we will also only focus on a subset of pollen variables. Namely, Erle, Gräser, Hasel and Birke

In [ ]:
#Drop not relevant columns
ts_df = ts_df.drop(columns=["sunshine_duration", "buche_pollen", "gemeine_esche_pollen", "eiche_pollen"])
print(ts_df.columns)

In [ ]:
#Create df for environmental and pollen variables
environ_df = ts_df[['temp_mean', 'temp_max', 'temp_min', 'soil_temp_5cm', 'rel_humidity', 'tot_precipitation', 'windspeed', 'glob_radiation']]
pollen_df = ts_df[['erle_pollen', 'birke_pollen', 'hasel_pollen', 'graeser_pollen']]

### 2. Plot and Analyze the Structure

In [ ]:
#Plot environmental variables
axes = environ_df.plot(subplots=True, figsize=(14, 12), sharex=True)

for ax in axes:
    ax.legend(loc="lower right")
    ax.grid(True)

plt.tight_layout()
plt.show()

temp_mean: no observable trend, annual periodicity, some slight outliers\
temp_max: no observable trend, annual periodicity, some slight outliers\
temp_min: no observable trend, annual periodicity, some slight outliers\
soil_temp_5cm: no oservable trend, annual periodicity, one outlier in 2010\
rel_humidity: no observable trend, slight to no periodicty, very noisy\
tot_percipitation: very slight decreas trend, annual periodicity, some outliers\
windspeed: very slight decreas trend, annual periodicity, few outliers\
glob_radiation: no observable trend, annual periodicity, no outliers

In [ ]:
#Plot smoothed environmental variables
environ_rolling_df = pd.DataFrame(index=environ_df.index)
for col in environ_df.columns:
    environ_rolling_df[f"{col}_rolling_3yr"] = environ_df[col].rolling(3*365, center=True).mean()

axes = environ_rolling_df.plot(subplots=True, figsize=(14, 12), sharex=True)
for ax in axes:
    ax.legend(loc="lower right")
    ax.grid(True)

plt.tight_layout()
plt.show()

temp_mean: We see an increase of about 2 degrees\
temp_min: same\
temp_max: same\
soil_temp_5cm: Drops arround 2008 but recover starts around 2018\
rel_humidity: Decreased\
tot_percipitation: stays around 3 
windspeed: shows low plateau abetween 2007 and 2009, increase around 2020 to 2025
glob_radiation: Medium increase

In [ ]:
#Plot pollen variables
axes = pollen_df.plot(subplots=True, figsize=(14, 12), sharex=True)

for ax in axes:
    ax.legend(loc="upper right")
    ax.grid(True)

plt.tight_layout()
plt.show()

erle_pollen: no trend, annual periodicity, spiky\
birke_pollen: seems to increase, annual periodicity, spiky\
hasel_pollen: no trend, mostly annual periodicity, not really peresent between 1997 and 2005, spiky\
graser_pollen:slight increase, annual periodicity, smoother, slight outlier in 2015 and 2024

In [ ]:
#Plot smoothed pollen variables
pollen_rolling_df = pd.DataFrame(index=pollen_df.index)
for col in pollen_df.columns:
    pollen_rolling_df[f"{col}_rolling_3yr"] = pollen_df[col].rolling(3*365, center=True).mean()

axes = pollen_rolling_df.plot(subplots=True, figsize=(14, 8), sharex=True)

for ax in axes:
    ax.legend(loc="upper right")
    ax.grid(True)

plt.tight_layout()
plt.show()

erle_pollen: Stays approx the same\
birke_pollen: Shows increase in recent years (2020 onwards)\
hasel_pollen: Sharp incrase after 2005\
graeser_pollen: Also increased after 2020

We will now use a Mann-Kendall trend test to determine if we have statistical evidence for an increase or decrease.

In [ ]:
#Mann Kendall to test for increase/decrease
!pip install pymannkendall
import pymannkendall as mk

print(f"{'variable':<18} {'p-value':>12}  {'trend':<12} {'slope':>14}")
print("-" * 60)

for col in environ_df.columns:
    res = mk.original_test(environ_df[col])
    print(f"{col:<18} {res.p:>12.5f}  {str(res.trend):<12} {res.slope:>14.8f}")

All temps have increased according to the test. However, soiltemp has decreased, so did relative humidity. Total percipitation did not show trend. Windspeed is trending downwards, while global radiation is increasing. However, the slopes are quite small for all variables.

In [ ]:
#Mann Kendall to test for increase/decrease
print(f"{'variable':<18} {'p-value':>12}  {'trend':<12} {'slope':>14}")
print("-" * 60)
for col in pollen_df.columns:
    res = mk.original_test(pollen_df[col])
    print(f"{col:<18} {res.p:>12.5f}  {str(res.trend):<12} {res.slope:>14.8f}")

All Pollen types are increasing according to the test, however the slope is extremely small

## Trend, Seasonal and Residual Decomposition

Again, we will only focus on a subsets of variables here. (Others are highly correlated, like temp_mean and temp_max)

In [ ]:
#Variable Selection
sarima_df = ts_df[["temp_mean", "soil_temp_5cm", "tot_precipitation", "glob_radiation", 
                   "erle_pollen", "birke_pollen", "hasel_pollen", "graeser_pollen"]]

In [ ]:
#Plot ACF and PACF for differenced timeseries
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

nrow = len(sarima_df.columns)
ncol = 2

fig, axes = plt.subplots(nrow, ncol, sharex=True, figsize=(12, 3*nrow))

for i, col in enumerate(sarima_df.columns):
    y_diff = sarima_df[col].diff().diff(365).dropna()

    plot_acf(y_diff, lags=20, ax=axes[i][0])
    axes[i][0].set_title(f"ACF: {col}")

    plot_pacf(y_diff, lags=50, ax=axes[i][1], method="ywm")
    axes[i][1].set_title(f"PACF: {col}")


### Temp_mean ARIMA model

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

#Difference the time series
y = sarima_df["temp_mean"].dropna()
#fit model
model = SARIMAX(
    y,
    order=(2,1,2),
    seasonal_order=(0,1,0,365),
    enforce_stationarity=False,
    enforce_invertibility=False
)
res = model.fit()

In [ ]:
print(res.summary())

In [ ]:
residuals = res.resid

plt.figure(figsize=(12,4))
plt.plot(residuals)
plt.title("Residuals")
plt.show()

plot_acf(residuals)
plt.show()

from statsmodels.stats.diagnostic import acorr_ljungbox
print(acorr_ljungbox(res.resid.dropna(), lags=[10,20,30,40], return_df=True))

- All Perdictors are statistically significant
- Ljung Box for 1 Lag (Q) is 0.0, hence there is no 1 Lag Correlation (great)
- Residual Plot looks like white noise around 0
- ACF stays low for all lags ≥ 1
- Ljung box for more lags howevery show that there is some correlation left (Can come from large nb of observations)
- Overall, good model, explains temporal dependce quite well

### Soil Temp 5cm

In [ ]:
y = sarima_df["soil_temp_5cm"].diff().diff(365).dropna()
model = SARIMAX(y, order = (2, 0, 2))
res = model.fit()

In [ ]:
print(res.summary())    

In [ ]:
residuals = res.resid

plt.figure(figsize=(12,4))
plt.plot(residuals)
plt.title("Residuals")
plt.show()

plot_acf(residuals)
plt.show()
print(acorr_ljungbox(res.resid.dropna(), lags=[10,20,30,40], return_df=True))

- Again, all predictors are statistically significant
- 1 Lag Ljung Box looks good again
- Residual plot is bad, we see that the model did not capture some seasonality, differencing is not enough
- Autocorrelation plot is fine
- Box Ljung for other lags show that residuals are not white noise

We will try an STL decomposition approach

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(sarima_df["soil_temp_5cm"], period=365)
decomp = stl.fit()
y = decomp.resid

model = SARIMAX(y, order = (2, 0, 2))
res = model.fit()

In [ ]:
print(res.summary())

In [ ]:
residuals = res.resid

plt.figure(figsize=(12,4))
plt.plot(residuals)
plt.title("Residuals")
plt.show()

plot_acf(residuals)
plt.show()
print(acorr_ljungbox(res.resid.dropna(), lags=[10,20,30,40], return_df=True))

- One LAg BoxLjung slightly increased
- Residuals still show seasonal dependence
- Autocorrelation is fine
- Box-Ljung for other lags has increased a bit which is better

ARMA-type modelling after standard processing does not fully capture the temporal structure.
This suggest the usage of a more complex model ? 

### Lets do the same thing for percipitation

In [ ]:
y = sarima_df["tot_precipitation"].diff().diff(365).dropna()

model = SARIMAX(y, order=(2, 0, 2))
res = model.fit()

In [ ]:
print(res.summary())

In [ ]:
residuals = res.resid

plt.figure(figsize=(12,4))
plt.plot(residuals)
plt.title("Residuals")
plt.show()

plot_acf(residuals)
plt.show()
print(acorr_ljungbox(res.resid.dropna(), lags=[10,20,30,40], return_df=True))

- After differencing, the ARMA(2,2) model provides a good fit for the total precipitation series.
- Residual plot still shows some non white noise behaviour.
- The residual Ljung Box tests at lags 10, 20, 30, and 40 are not significant.
- Therefore, the model captures the main temporal dependence structure of the differenced series well.
- Most model coefficients are significant, although the MA(1) term is not significant, suggesting that a simpler model could also be considered.
- This is not unusual for precipitation data, which often show heavy tails and extreme values.
- Overall, the model appears adequate for describing the differenced precipitation series.